# Librerías por practicidad (numpy con matrices, y warnings para limpiar output)

In [14]:
import numpy as np
import warnings

# Clases y métodos para los algoritmos

In [ ]:
class SolucionadoresLineales:
    
    @staticmethod # Sugerido por IA tener los métodos estátcos.
    def eliminacion_gaussiana_pivoteo(A, b):
        """Resuelve Ax = b mediante eliminación gaussiana con pivoteo parcial de renglón."""
        n = len(b)
        # Se crea la matriz aumentada [A|b] evitando modificar las originales
        M = np.concatenate((A, b.reshape(n, 1)), axis=1).astype(float)

        # Eliminación hacia adelante
        for i in range(n):
            # Pivoteo parcial: encontrar la fila con el valor máximo en la columna actual
            max_row = np.argmax(abs(M[i:, i])) + i 
            if i != max_row:
                # Intercambio de renglones
                M[[i, max_row]] = M[[max_row, i]]

            if M[i, i] == 0:
                raise ValueError("El sistema no tiene solución única (pivote cero).")

            # Hacer ceros debajo del pivote
            for j in range(i + 1, n):
                factor = M[j, i] / M[i, i]
                M[j, i:] -= factor * M[i, i:]

        # Sustitución hacia atrás
        x = np.zeros(n)
        for i in range(n - 1, -1, -1):
            # Se calcula el valor despejando la incógnita usando los valores ya encontrados
            suma_conocidos = sum(M[i, j] * x[j] for j in range(i + 1, n))
            x[i] = (M[i, n] - suma_conocidos) / M[i, i]

        return x

    @staticmethod
    def tridiagonal_lu(A, b):
        """
        Resuelve Ax = b mediante Factorización LU para matrices tridiagonales. 
        Asume que A es tridiagonal.
        """
        n = len(b)
        
        # Extracción de las diagonales de la matriz A
        a = np.diag(A, k=-1).astype(float)  # Diagonal inferior
        d = np.diag(A, k=0).astype(float)   # Diagonal principal
        c = np.diag(A, k=1).astype(float)   # Diagonal superior
        v = b.copy().astype(float)

        # Eliminación hacia adelante (Modificando las diagonales)
        for i in range(1, n):
            factor = a[i-1] / d[i-1]
            d[i] -= factor * c[i-1]
            v[i] -= factor * v[i-1]

        # Sustitución hacia atrás
        x = np.zeros(n)
        x[-1] = v[-1] / d[-1]
        for i in range(n - 2, -1, -1):
            x[i] = (v[i] - c[i] * x[i + 1]) / d[i]

        return x

    @staticmethod
    def jacobi(A, b, tol=1e-5, max_iter=1000):
        """Resuelve Ax = b mediante el método iterativo de Jacobi."""
        n = len(b)
        x = np.zeros(n)
        x_new = np.zeros(n)

        for iteracion in range(max_iter):
            for i in range(n):
                # Sumatoria de A[i,j] * x[j] excluyendo la diagonal
                suma = sum(A[i, j] * x[j] for j in range(n) if j != i)
                x_new[i] = (b[i] - suma) / A[i, i]

            # Criterio de convergencia usando la norma infinito (máxima diferencia absoluta)
            if np.max(np.abs(x_new - x)) < tol:
                return x_new, iteracion + 1

            # Actualizamos el vector x para la siguiente iteración
            x = x_new.copy()

        raise ValueError("El método de Jacobi no convergió en el máximo de iteraciones.")

    @staticmethod
    def gauss_seidel(A, b, tol=1e-5, max_iter=1000):
        """Resuelve Ax = b mediante el método iterativo de Gauss-Seidel."""
        n = len(b)
        x = np.zeros(n)

        for iteracion in range(max_iter):
            x_old = x.copy()

            for i in range(n):
                # En Gauss-Seidel usamos los valores de 'x' que ya fueron calculados en esta misma iteración
                suma_actualizados = sum(A[i, j] * x[j] for j in range(i))
                suma_anteriores = sum(A[i, j] * x_old[j] for j in range(i + 1, n))
                
                x[i] = (b[i] - suma_actualizados - suma_anteriores) / A[i, i]

            # Criterio de convergencia
            if np.max(np.abs(x - x_old)) < tol:
                return x, iteracion + 1

        raise ValueError("El método de Gauss-Seidel no convergió en el máximo de iteraciones.")

# Primera matriz A y vector b:

In [15]:
# PRIMERA PARTE

# Ignorar las advertencias de overflow y cálculo inválido de numpy
warnings.filterwarnings('ignore', category=RuntimeWarning)
print("=== PARTE 1: A1 ===")

A1 = np.array([
    [1, 2, -3, 4, 5],
    [-2, -5, 8, -8, -9],
    [1, 2, -2, 7, 9],
    [1, 1, 0, 6, 12],
    [2, 4, -6, 8, 11]
], dtype=float)
b1 = np.array([-1, -2, 4, 6, -3], dtype=float)

# Método Directo
x_gauss_1 = SolucionadoresLineales.eliminacion_gaussiana_pivoteo(A1, b1)
print(f"Método Directo (Gauss con pivoteo): {x_gauss_1}")

# Métodos Iterativos (con manejo de excepciones)
try:
    x_jacobi_1, iter_j_1 = SolucionadoresLineales.jacobi(A1, b1, tol=1e-5)
    print(f"Iterativo (Jacobi):                 {x_jacobi_1} -> Convergió en {iter_j_1} iteraciones.")
except ValueError as e:
    print(f"Iterativo (Jacobi):                 [DIVERGIÓ] {e}") # Como la matriz debe de ser diagonalmente dominante, metemos esta excepción.

try:
    x_gs_1, iter_gs_1 = SolucionadoresLineales.gauss_seidel(A1, b1, tol=1e-5)
    print(f"Iterativo (Gauss-Seidel):           {x_gs_1} -> Convergió en {iter_gs_1} iteraciones.")
except ValueError as e:
    print(f"Iterativo (Gauss-Seidel):           [DIVERGIÓ] {e}") # Como la matriz debe de ser diagonalmente dominante, metemos esta excepción, al igual que en el método de Jacobi.



=== PARTE 1: A1 ===
Método Directo (Gauss con pivoteo): [-3. 69. 33. -8. -1.]
Iterativo (Jacobi):                 [DIVERGIÓ] El método de Jacobi no convergió en el máximo de iteraciones.
Iterativo (Gauss-Seidel):           [DIVERGIÓ] El método de Gauss-Seidel no convergió en el máximo de iteraciones.


# Segunda Matriz A y Vector b:

In [16]:
# SEGUNDA PARTE
print("\n=== PARTE 2: A2 ===")

A2 = np.array([
    [10, 5, 0, 0],
    [5, 10, -1, 0],
    [0, -4, 8, -1],
    [0, 0, -1, 5]
], dtype=float)
b2 = np.array([6, 25, -11, -11], dtype=float)

# Método Directo
x_gauss_2 = SolucionadoresLineales.eliminacion_gaussiana_pivoteo(A2, b2)
x_lu_tridiag = SolucionadoresLineales.tridiagonal_lu(A2, b2)
print(f"Método Directo (Gauss con pivoteo): {x_gauss_2}")
print(f"Método Directo (LU Tridiagonal):    {x_lu_tridiag}")

# Métodos Iterativos (Para A2 sí van a converger, por la dominancia diagonal)
try:
    x_jacobi_2, iter_j_2 = SolucionadoresLineales.jacobi(A2, b2, tol=1e-5)
    print(f"Iterativo (Jacobi):                 {x_jacobi_2} -> Convergió en {iter_j_2} iteraciones.")
    
    x_gs_2, iter_gs_2 = SolucionadoresLineales.gauss_seidel(A2, b2, tol=1e-5)
    print(f"Iterativo (Gauss-Seidel):           {x_gs_2} -> Convergió en {iter_gs_2} iteraciones.")
except ValueError as e:
    print(f"Error: {e}")


=== PARTE 2: A2 ===
Método Directo (Gauss con pivoteo): [-0.85321101  2.90642202 -0.20183486 -2.24036697]
Método Directo (LU Tridiagonal):    [-0.85321101  2.90642202 -0.20183486 -2.24036697]
Iterativo (Jacobi):                 [-0.85320965  2.90641626 -0.20183635 -2.24036903] -> Convergió en 22 iteraciones.
Iterativo (Gauss-Seidel):           [-0.85320897  2.90642077 -0.20183554 -2.24036711] -> Convergió en 12 iteraciones.
